# Machine Learning Modeling — Advanced Reference
> **Level:** Advanced | **Goal:** End-to-end ML pipelines, evaluation, tuning, and interpretability

## Table of Contents
1. [Scikit-learn Pipelines](#pipelines)
2. [Feature Engineering & Selection](#features)
3. [Cross-Validation Strategies](#cv)
4. [Hyperparameter Tuning](#tuning)
5. [Model Evaluation — Full Metrics Suite](#evaluation)
6. [Ensemble Methods](#ensembles)
7. [Model Interpretability (SHAP)](#shap)
8. [Class Imbalance](#imbalance)
9. [Gradient Boosting (XGBoost / LightGBM)](#boosting)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, KFold,
    GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, LabelEncoder,
    PolynomialFeatures, PowerTransformer
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif,
    RFE, SelectFromModel, RFECV
)
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, AdaBoostClassifier,
    VotingClassifier, StackingClassifier, BaggingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, log_loss,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay

rng = np.random.default_rng(42)
print("Setup complete")

---
## 1 · Scikit-learn Pipelines <a id='pipelines'></a>

Pipelines chain preprocessing + model into one object:
- **No data leakage** — transformers are fit only on training folds
- **Single `.fit()` / `.predict()` interface**
- **Hyperparameter tuning across all steps simultaneously**

In [ ]:
# ── Build a realistic dataset with mixed types ─────────────────
n = 2000
df = pd.DataFrame({
    'age':        rng.integers(18, 70, n).astype(float),
    'income':     rng.lognormal(10, 1, n),
    'debt_ratio': rng.beta(2, 5, n),
    'n_products':  rng.integers(0, 10, n).astype(float),
    'region':     rng.choice(['North','South','East','West'], n),
    'job_type':   rng.choice(['employed','self_employed','retired','student'], n),
})

# Inject missing values
for col, rate in [('age', 0.05), ('income', 0.08), ('region', 0.03)]:
    mask = rng.random(n) < rate
    df.loc[mask, col] = np.nan

# Target: default (binary)
logit = (-3 + 0.02*df['age'].fillna(40) - 0.3*df['income'].fillna(df['income'].median())/10000
         + 3*df['debt_ratio'] + rng.normal(0, 0.5, n))
y = (1 / (1 + np.exp(-logit)) > 0.5).astype(int)

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class balance: {y_train.mean():.2%} positive")

In [ ]:
# ── ColumnTransformer + Pipeline ──────────────────────────────
numeric_features  = ['age', 'income', 'debt_ratio', 'n_products']
categorical_features = ['region', 'job_type']

numeric_transformer = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler',  RobustScaler()),          # robust to outliers
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,  numeric_features),
    ('cat', categorical_transformer, categorical_features),
], remainder='drop')

pipeline = Pipeline([
    ('prep',   preprocessor),
    ('model',  LogisticRegression(max_iter=1000, class_weight='balanced'))
])

pipeline.fit(X_train, y_train)
y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_proba):.4f}")
print("\n", classification_report(y_test, y_pred))

---
## 2 · Feature Engineering & Selection <a id='features'></a>

In [ ]:
# ── Interaction features with PolynomialFeatures ───────────────
from sklearn.pipeline import Pipeline

poly_pipeline = Pipeline([
    ('prep',   preprocessor),
    ('poly',   PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('select', SelectKBest(mutual_info_classif, k=20)),
    ('model',  LogisticRegression(max_iter=1000, class_weight='balanced'))
])

cv_scores = cross_val_score(poly_pipeline, X_train, y_train,
                            cv=StratifiedKFold(5), scoring='roc_auc')
print(f"Poly+SelectKBest CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# ── Feature importance from tree models ───────────────────────
rf_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))
])
rf_pipeline.fit(X_train, y_train)

# Get feature names from ColumnTransformer
cat_names = rf_pipeline.named_steps['prep'].named_transformers_['cat']['encoder'].get_feature_names_out(categorical_features)
feature_names = numeric_features + list(cat_names)
importances = rf_pipeline.named_steps['model'].feature_importances_

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.head(12).plot.barh(ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 12 Feature Importances (Random Forest)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3 · Cross-Validation Strategies <a id='cv'></a>

| Strategy | When to use |
|---|---|
| `KFold(k=5)` | Regression, balanced classes |
| `StratifiedKFold(k=5)` | Classification — preserves class ratio |
| `TimeSeriesSplit` | Time series — respect temporal order |
| `GroupKFold` | Groups must not span folds (user-level data) |
| `RepeatedStratifiedKFold` | Small datasets — reduce variance of estimate |

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GroupKFold, RepeatedStratifiedKFold

# ── Compare CV strategies ──────────────────────────────────────
strategies = {
    'KFold-5':                 KFold(n_splits=5, shuffle=True, random_state=42),
    'StratifiedKFold-5':       StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'RepeatedStratifiedKFold': RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42),
}

# Preprocess once for speed
X_preprocessed = preprocessor.fit_transform(X_train, y_train)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
for name, cv in strategies.items():
    scores = cross_val_score(model, X_preprocessed, y_train, cv=cv, scoring='roc_auc')
    print(f"{name:35s}: {scores.mean():.4f} ± {scores.std():.4f}  (n_evals={len(scores)})")

In [ ]:
# ── Learning curves: detect bias/variance ─────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    rf_pipeline, X_train, y_train,
    cv=StratifiedKFold(5),
    scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_scores.mean(1), 'o-', label='Train', color='steelblue')
ax.fill_between(train_sizes, train_scores.mean(1)-train_scores.std(1),
                train_scores.mean(1)+train_scores.std(1), alpha=0.15, color='steelblue')
ax.plot(train_sizes, val_scores.mean(1), 'o-', label='Validation', color='crimson')
ax.fill_between(train_sizes, val_scores.mean(1)-val_scores.std(1),
                val_scores.mean(1)+val_scores.std(1), alpha=0.15, color='crimson')
ax.set(title='Learning Curves (Random Forest)', xlabel='Training size',
       ylabel='ROC-AUC')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.show()

---
## 4 · Hyperparameter Tuning <a id='tuning'></a>

In [ ]:
from scipy.stats import randint, uniform

# ── RandomizedSearchCV (faster than Grid for large spaces) ────
param_dist = {
    'model__n_estimators':     randint(100, 500),
    'model__max_depth':        [None, 5, 10, 15, 20],
    'model__min_samples_leaf': randint(1, 20),
    'model__max_features':     ['sqrt', 'log2', 0.5],
    'model__class_weight':     ['balanced', 'balanced_subsample']
}

search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=StratifiedKFold(3),
    scoring='roc_auc',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=0
)
search.fit(X_train, y_train)

print(f"Best CV ROC-AUC: {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

best_model = search.best_estimator_
y_pred_best  = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]
print(f"\nTest ROC-AUC: {roc_auc_score(y_test, y_proba_best):.4f}")

---
## 5 · Model Evaluation — Full Metrics Suite <a id='evaluation'></a>

In [ ]:
# ── Comprehensive evaluation function ─────────────────────────
def evaluate_classifier(model, X_test, y_test, model_name='Model', threshold=0.5):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)

    metrics = {
        'Accuracy':        accuracy_score(y_test, y_pred),
        'Precision':       precision_score(y_test, y_pred),
        'Recall':          recall_score(y_test, y_pred),
        'F1':              f1_score(y_test, y_pred),
        'ROC-AUC':         roc_auc_score(y_test, y_proba),
        'PR-AUC':          average_precision_score(y_test, y_proba),
        'Log-Loss':        log_loss(y_test, y_proba),
    }
    print(f"── {model_name} ──")
    for name, val in metrics.items():
        print(f"  {name:15s}: {val:.4f}")
    return metrics

evaluate_classifier(best_model, X_test, y_test, 'Random Forest (Tuned)')

In [ ]:
# ── Visual evaluation: 4-panel diagnostic ─────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# 1. Confusion Matrix
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test,
                                       normalize='true', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix (normalized)')

# 2. ROC Curve
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1])
axes[1].set_title('ROC Curve')

# 3. Precision-Recall Curve
PrecisionRecallDisplay.from_estimator(best_model, X_test, y_test, ax=axes[2])
axes[2].set_title('Precision-Recall Curve')

# 4. Calibration Plot
CalibrationDisplay.from_estimator(best_model, X_test, y_test,
                                   n_bins=10, ax=axes[3])
axes[3].set_title('Calibration Curve')

plt.suptitle('Model Evaluation Dashboard', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6 · Ensemble Methods <a id='ensembles'></a>

In [ ]:
# ── Stacking Classifier ────────────────────────────────────────
base_estimators = [
    ('lr',  LogisticRegression(max_iter=1000, class_weight='balanced')),
    ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ('svm', SVC(probability=True, kernel='rbf', class_weight='balanced', random_state=42)),
]
meta_learner = LogisticRegression(max_iter=500)

stacking_pipeline = Pipeline([
    ('prep', preprocessor),
    ('stack', StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_learner,
        cv=StratifiedKFold(5),
        passthrough=False,
        stack_method='predict_proba'
    ))
])

stacking_pipeline.fit(X_train, y_train)
y_proba_stack = stacking_pipeline.predict_proba(X_test)[:, 1]
print(f"Stacking ROC-AUC: {roc_auc_score(y_test, y_proba_stack):.4f}")

---
## 7 · Model Interpretability (SHAP) <a id='shap'></a>

SHAP (SHapley Additive exPlanations) decomposes each prediction into feature contributions.

- `TreeExplainer` — fast for tree-based models
- `LinearExplainer` — for linear models
- `KernelExplainer` — model-agnostic (slow)

In [ ]:
try:
    import shap

    # Extract fitted RF from pipeline
    rf_model = best_model.named_steps['model']
    X_test_transformed = best_model.named_steps['prep'].transform(X_test)

    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test_transformed)

    # Global importance
    shap.summary_plot(shap_values[1], X_test_transformed,
                      feature_names=feature_names, show=True)

    # Single prediction explanation
    idx = 0
    shap.waterfall_plot(shap.Explanation(
        values=shap_values[1][idx],
        base_values=explainer.expected_value[1],
        data=X_test_transformed[idx],
        feature_names=feature_names
    ))

except ImportError:
    print("Install shap: pip install shap")
    print("SHAP provides global and local model explanations.")
    print("Key plots:")
    print("  shap.summary_plot()   — global feature importance beeswarm")
    print("  shap.waterfall_plot() — single prediction explanation")
    print("  shap.dependence_plot()— feature interaction with another")

---
## 8 · Class Imbalance <a id='imbalance'></a>

In [ ]:
# ── Strategies for imbalanced data ────────────────────────────
print("Strategies for class imbalance:")
print("  1. class_weight='balanced' in sklearn models")
print("  2. SMOTE/ADASYN (oversample minority class) — via imbalanced-learn")
print("  3. Undersampling majority class")
print("  4. Adjust decision threshold")
print("  5. Use PR-AUC / F1 metrics instead of accuracy")

# ── Threshold optimization (F1 maximization) ─────────────────
from sklearn.metrics import precision_recall_curve

y_proba_rf = best_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_proba_rf)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f"\nDefault threshold (0.5):")
print(classification_report(y_test, (y_proba_rf >= 0.5).astype(int)))

print(f"Optimal threshold ({best_threshold:.3f}):")
print(classification_report(y_test, (y_proba_rf >= best_threshold).astype(int)))

---
## 9 · Gradient Boosting: XGBoost / LightGBM <a id='boosting'></a>

### Key hyperparameters

| Parameter | Effect | Typical range |
|---|---|---|
| `n_estimators` | # of trees | 100–1000 (use early stopping) |
| `learning_rate` | Step size | 0.01–0.3 (lower = more trees needed) |
| `max_depth` | Tree depth | 3–8 |
| `subsample` | Row sampling | 0.6–1.0 |
| `colsample_bytree` | Column sampling | 0.6–1.0 |
| `min_child_weight` | Regularization | 1–10 |
| `reg_alpha` / `reg_lambda` | L1/L2 reg | 0–10 |

In [ ]:
try:
    import xgboost as xgb
    import lightgbm as lgb

    X_tr_prep = preprocessor.fit_transform(X_train, y_train)
    X_te_prep = preprocessor.transform(X_test)

    # ── XGBoost with early stopping ────────────────────────────
    xgb_model = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train==0).sum() / (y_train==1).sum(),  # handle imbalance
        eval_metric='auc',
        early_stopping_rounds=20,
        random_state=42
    )
    X_trn, X_val, y_trn, y_val = train_test_split(X_tr_prep, y_train, test_size=0.15, random_state=0)
    xgb_model.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], verbose=0)

    y_proba_xgb = xgb_model.predict_proba(X_te_prep)[:, 1]
    print(f"XGBoost ROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.4f}  (best iter={xgb_model.best_iteration})")

    # ── LightGBM ───────────────────────────────────────────────
    lgb_model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        is_unbalance=True,
        random_state=42,
        verbose=-1
    )
    lgb_model.fit(X_trn, y_trn,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(20, verbose=False)])

    y_proba_lgb = lgb_model.predict_proba(X_te_prep)[:, 1]
    print(f"LightGBM ROC-AUC: {roc_auc_score(y_test, y_proba_lgb):.4f}")

except ImportError as e:
    print(f"Install: pip install xgboost lightgbm\nError: {e}")
    print("\nXGBoost/LightGBM are go-to boosting libraries:")
    print("  - Faster than sklearn GradientBoosting")
    print("  - Native support for early stopping & missing values")
    print("  - LightGBM is faster on large datasets (leaf-wise vs depth-wise)")

---
## 🏥 Project — Airline Medical Kit: Predict Kit Usefulness <a id='airline-medical'></a>

### Business context
An airline launched a medical kit sold during check-in. After collecting data from passengers who bought it, they want to predict **whether the kit will actually be useful** to a given passenger — so they can better target marketing and improve kit contents.

### Target variable
`kit_used` — binary:
- `1` = passenger used the kit during the flight
- `0` = passenger bought it but did not use it

### Features

| Feature | Type | Description |
|---|---|---|
| `age` | Numeric | Passenger age |
| `flight_duration_h` | Numeric | Flight duration in hours |
| `travel_class` | Categorical | Economy / Business / First |
| `has_chronic_condition` | Binary | Known chronic condition (yes/no) |
| `previous_kit_purchases` | Numeric | # of times bought the kit before |
| `altitude_sensitivity` | Numeric | Self-reported sensitivity score (1–10) |
| `flight_type` | Categorical | Domestic / International |
| `seat_position` | Categorical | Window / Middle / Aisle |
| `meal_type` | Categorical | Standard / Vegetarian / Diabetic / Low-sodium |

### Task
Binary classification → predict `kit_used`

In [ ]:
# ── Step 1: Generate the airline medical kit dataset ───────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                             ConfusionMatrixDisplay, RocCurveDisplay,
                             PrecisionRecallDisplay, average_precision_score)

rng = np.random.default_rng(0)
n = 3000

age                    = rng.integers(18, 80, n).astype(float)
flight_duration_h      = rng.uniform(0.5, 16, n)
has_chronic_condition  = rng.integers(0, 2, n)
previous_kit_purchases = rng.integers(0, 6, n).astype(float)
altitude_sensitivity   = rng.integers(1, 11, n).astype(float)
travel_class           = rng.choice(['Economy', 'Business', 'First'], n, p=[0.7, 0.2, 0.1])
flight_type            = rng.choice(['Domestic', 'International'], n, p=[0.45, 0.55])
seat_position          = rng.choice(['Window', 'Middle', 'Aisle'], n)
meal_type              = rng.choice(['Standard', 'Vegetarian', 'Diabetic', 'Low-sodium'], n,
                                    p=[0.55, 0.2, 0.15, 0.1])

# Inject missing values
age[rng.random(n) < 0.04] = np.nan
flight_duration_h[rng.random(n) < 0.03] = np.nan
altitude_sensitivity[rng.random(n) < 0.05] = np.nan

# Target: logistic function of meaningful signals
logit = (
    -2.0
    + 0.015 * np.where(np.isnan(age), 40, age)
    + 0.12  * np.where(np.isnan(flight_duration_h), 5, flight_duration_h)
    + 1.2   * has_chronic_condition
    + 0.35  * previous_kit_purchases
    + 0.10  * np.where(np.isnan(altitude_sensitivity), 5, altitude_sensitivity)
    + 0.4   * (meal_type == 'Diabetic').astype(int)
    + 0.3   * (meal_type == 'Low-sodium').astype(int)
    + 0.2   * (flight_type == 'International').astype(int)
    + rng.normal(0, 0.4, n)
)
prob     = 1 / (1 + np.exp(-logit))
kit_used = (prob > 0.5).astype(int)

df = pd.DataFrame({
    'age':                    age,
    'flight_duration_h':      flight_duration_h,
    'travel_class':           travel_class,
    'has_chronic_condition':  has_chronic_condition,
    'previous_kit_purchases': previous_kit_purchases,
    'altitude_sensitivity':   altitude_sensitivity,
    'flight_type':            flight_type,
    'seat_position':          seat_position,
    'meal_type':              meal_type,
    'kit_used':               kit_used
})

print(f"Dataset: {df.shape}")
print(f"Target balance: {kit_used.mean():.1%} used kit")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum()>0]}")
display(df.head(8))

In [ ]:
# ── Step 2: EDA — understand the data ─────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# Numeric distributions by target
for ax, col in zip(axes[0], ['age', 'flight_duration_h', 'altitude_sensitivity', 'previous_kit_purchases']):
    for label, grp in df.groupby('kit_used'):
        ax.hist(grp[col].dropna(), bins=25, alpha=0.6,
                label=f"kit_used={label}")
    ax.set_title(col)
    ax.legend(fontsize=7)
    ax.spines[['top','right']].set_visible(False)

# Categorical rates
for ax, col in zip(axes[1], ['travel_class', 'flight_type', 'meal_type', 'has_chronic_condition']):
    rates = df.groupby(col)['kit_used'].mean().sort_values()
    ax.barh(rates.index, rates.values, color='steelblue')
    ax.axvline(df['kit_used'].mean(), color='crimson', ls='--', label='overall mean')
    ax.set_title(f'Kit use rate by {col}')
    ax.set_xlabel('P(kit used)')
    ax.legend(fontsize=7)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('EDA — Airline Medical Kit Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n── Correlation with target ──")
corr = df.select_dtypes(include='number').corr()['kit_used'].drop('kit_used').sort_values()
print(corr.round(3))

In [ ]:
# ── Step 3: Build ML pipeline & compare models ─────────────────
X = df.drop(columns='kit_used')
y = df['kit_used']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# ── Preprocessing ──────────────────────────────────────────────
num_cols = ['age', 'flight_duration_h', 'altitude_sensitivity',
            'previous_kit_purchases', 'has_chronic_condition']
cat_cols = ['travel_class', 'flight_type', 'seat_position', 'meal_type']

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
])

# ── Three models ───────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    pipe.fit(X_train, y_train)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_proba)
    results[name] = {'pipeline': pipe, 'test_auc': test_auc,
                     'cv_mean': cv_auc.mean(), 'cv_std': cv_auc.std()}
    print(f"{name:25s}  CV-AUC={cv_auc.mean():.4f}±{cv_auc.std():.4f}  Test-AUC={test_auc:.4f}")

best_name = max(results, key=lambda k: results[k]['test_auc'])
best_pipe  = results[best_name]['pipeline']
print(f"\n✅ Best model: {best_name}")

# ── Evaluation plots ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

y_best_pred  = best_pipe.predict(X_test)
y_best_proba = best_pipe.predict_proba(X_test)[:, 1]

ConfusionMatrixDisplay.from_estimator(best_pipe, X_test, y_test,
                                       normalize='true', cmap='Blues', ax=axes[0])
axes[0].set_title(f'Confusion Matrix\n{best_name}')

RocCurveDisplay.from_estimator(best_pipe, X_test, y_test, ax=axes[1])
axes[1].set_title(f'ROC Curve — AUC={roc_auc_score(y_test, y_best_proba):.3f}')

PrecisionRecallDisplay.from_estimator(best_pipe, X_test, y_test, ax=axes[2])
axes[2].set_title(f'Precision-Recall — AP={average_precision_score(y_test, y_best_proba):.3f}')

plt.suptitle(f'Best Model: {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("\n── Classification Report ──")
print(classification_report(y_test, y_best_pred, target_names=['Not Used', 'Used']))

In [ ]:
# ── Step 4: Feature importance ─────────────────────────────────
import warnings; warnings.filterwarnings('ignore')

# Get feature names after preprocessing
cat_names = (best_pipe.named_steps['prep']
             .named_transformers_['cat']['encoder']
             .get_feature_names_out(cat_cols))
feature_names = num_cols + list(cat_names)

# Works for RF and GB — both have feature_importances_
try:
    importances = best_pipe.named_steps['model'].feature_importances_
    feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    feat_imp.head(15).plot.barh(ax=ax, color='steelblue')
    ax.invert_yaxis()
    ax.set_title(f'Top 15 Feature Importances — {best_name}', fontweight='bold')
    ax.set_xlabel('Importance')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()

    print("\n── Top 10 features ──")
    print(feat_imp.head(10).round(4))

except AttributeError:
    # LogisticRegression: use coefficients
    coef = best_pipe.named_steps['model'].coef_[0]
    feat_imp = pd.Series(np.abs(coef), index=feature_names).sort_values(ascending=False)
    print("Logistic Regression — top features by |coefficient|:")
    print(feat_imp.head(10).round(4))

# ── Step 5: Business insight summary ──────────────────────────
print("\n" + "="*55)
print("  BUSINESS INSIGHTS — Who is most likely to USE the kit?")
print("="*55)
insights = [
    "✅ Passengers with chronic conditions → highest predictor",
    "✅ Longer flights → significantly more kit usage",
    "✅ Diabetic / Low-sodium meal → signals health awareness",
    "✅ Higher altitude sensitivity score → more kit usage",
    "✅ Repeat buyers → already found the kit useful before",
    "💡 Recommendation: target marketing to chronic + long-haul",
]
for i in insights:
    print(f"  {i}")

---
## 🏥 Real Project — Airline Medical Kit (train.csv / test.csv) <a id='airline-real'></a>

**Real dataset** from `dataset/train.csv` and `dataset/test.csv`.

| Column | Type | Description |
|---|---|---|
| `ID` | String | Unique passenger identifier |
| `Distributor` | Int | Sales distributor code |
| `Product` | Int | Product type |
| `Duration` | Int | Policy duration (days) |
| `Destination` | Int | Destination code |
| `Sales` | Float | Policy sales amount |
| `Commission` | Float | Commission paid |
| `Gender` | Float | 0 = Female, 1 = Male (**70% missing**) |
| `Age` | Int | Passenger age |
| `Target` | Int | **1 = claim filed**, 0 = no claim |

**Key challenge:** Severe class imbalance — only **4.69%** positive (316 out of 6,736).  
→ Use `class_weight="balanced"`, evaluate with ROC-AUC and PR-AUC (not accuracy).

In [ ]:
# ── Step 1: Load the real dataset ──────────────────────────────
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "dataset/"   # relative to notebook location

train = pd.read_csv(os.path.join(BASE, "train.csv"))
test  = pd.read_csv(os.path.join(BASE, "test.csv"))

print(f"Train shape : {train.shape}")
print(f"Test  shape : {test.shape}")
print(f"\nTarget balance: {train['Target'].value_counts().to_dict()}")
print(f"Positive rate : {train['Target'].mean():.2%}  ← highly imbalanced!")
print(f"\nMissing values in train:\n{train.isnull().sum()[train.isnull().sum()>0]}")
display(train.head(6))

In [ ]:
# ── Step 2: EDA ─────────────────────────────────────────────────
NUM_COLS = ["Distributor", "Product", "Duration", "Destination",
            "Sales", "Commission", "Gender", "Age"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, NUM_COLS):
    for label, grp in train.groupby("Target"):
        ax.hist(grp[col].dropna(), bins=30, alpha=0.65,
                label=f"Target={label}", density=True)
    ax.set_title(col, fontweight="bold")
    ax.legend(fontsize=7)
    ax.spines[["top","right"]].set_visible(False)

plt.suptitle("EDA — Distribution by Target (train.csv)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("── Correlation with Target ──")
corr = train[NUM_COLS + ["Target"]].corr()["Target"].drop("Target").sort_values()
print(corr.round(3))

In [ ]:
# ── Step 3: Pipeline + Model comparison ───────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)

X, y = train[NUM_COLS], train["Target"]
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

def make_pipe(model):
    prep = ColumnTransformer([
        ("n", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
        ]), NUM_COLS),
    ])
    return Pipeline([("prep", prep), ("model", model)])

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest":       RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42),
}

cv   = StratifiedKFold(5, shuffle=True, random_state=42)
best = {"name": None, "pipe": None, "auc": 0}

print("── Model comparison ──────────────────────────────────────")
for name, model in models.items():
    p      = make_pipe(model)
    cv_auc = cross_val_score(p, X_tr, y_tr, cv=cv, scoring="roc_auc")
    p.fit(X_tr, y_tr)
    vauc   = roc_auc_score(y_val, p.predict_proba(X_val)[:, 1])
    pr_auc = average_precision_score(y_val, p.predict_proba(X_val)[:, 1])
    print(f"  {name:25s}  CV={cv_auc.mean():.4f}±{cv_auc.std():.4f}  "
          f"Val-ROC={vauc:.4f}  Val-PR={pr_auc:.4f}")
    if vauc > best["auc"]:
        best = {"name": name, "pipe": p, "auc": vauc}

print(f"\n✅ Best: {best['name']}  (Val-AUC={best['auc']:.4f})")

In [ ]:
# ── Step 4: Evaluation dashboard ──────────────────────────────
pipe = best["pipe"]
y_proba = pipe.predict_proba(X_val)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ConfusionMatrixDisplay.from_estimator(pipe, X_val, y_val,
                                       normalize="true", cmap="Blues", ax=axes[0])
axes[0].set_title(f"Confusion Matrix (norm.)\n{best['name']}")

RocCurveDisplay.from_estimator(pipe, X_val, y_val, ax=axes[1])
axes[1].set_title(f"ROC Curve — AUC={roc_auc_score(y_val, y_proba):.3f}")

PrecisionRecallDisplay.from_estimator(pipe, X_val, y_val, ax=axes[2])
axes[2].set_title(f"Precision-Recall — AP={average_precision_score(y_val, y_proba):.3f}")

plt.suptitle(f"Evaluation — {best['name']} (real dataset)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(classification_report(y_val, pipe.predict(X_val),
                             target_names=["No Claim (0)", "Claim (1)"]))

In [ ]:
# ── Step 5: Feature importance ─────────────────────────────────
try:
    imp = pipe.named_steps["model"].feature_importances_
except AttributeError:
    imp = np.abs(pipe.named_steps["model"].coef_[0])

feat_imp = pd.Series(imp, index=NUM_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
feat_imp.plot.barh(ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_title(f"Feature Importances — {best['name']}", fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()
print(feat_imp.round(4))

# ── Step 6: Retrain on full train → generate submission ────────
pipe.fit(X, y)
pred = pipe.predict(test[NUM_COLS])

submission = pd.DataFrame({"ID": test["ID"], "Target": pred})
submission.to_csv(os.path.join(BASE, "submission.csv"), index=False)

print(f"\n── submission.csv ──")
print(f"Rows: {len(submission)}  |  Predicted claims: {pred.sum()} ({pred.mean():.2%})")
display(submission.head(10))

---
## 📝 Quiz Notes — Machine Learning Concepts <a id='quiz-ml'></a>

---

### Q1 · K-NN difficulties

> ✅ **All of these:** slow as data grows, curse of dimensionality, distance from all training cases

| Difficulty | Why |
|---|---|
| Slow as data grows | Lazy learner — computes distance to every training point at prediction time: $O(n \cdot d)$ |
| Curse of dimensionality | In high dimensions all points become equidistant → distances lose meaning |
| Distance from all training cases | Must compare test point to all $n$ points — expensive at scale |

**Mitigations:** KD-trees, Ball-trees, PCA/UMAP, feature scaling (mandatory).

---

### Q2 · R² = 0.85

> ✅ **85% of the variability in house prices can be explained by square footage**

$$R^2 = \frac{\text{variance explained}}{\text{total variance}} = 0.85$$

- ≠ "predicts correctly 85%" → that's accuracy (classification)
- ≠ "price increases by 85%" → that's the slope coefficient β₁
- ≠ "85% of points on the line" → that would be R² = 1.0

---

### Q3 · K-means objective

> ✅ **Group similar data points into K distinct clusters based on feature similarity**

$$\min \sum_{k=1}^{K} \sum_{x \in C_k} \|x - \mu_k\|^2$$

- K must be **set by the user** — K-means does NOT find it automatically
- Use Elbow method or Silhouette score to choose K

---

### Q4 · Accuracy calculation

> ✅ **90%** — 850 non-fraud + 50 fraud = 900 correct out of 1000

$$\text{Accuracy} = \frac{850 + 50}{1000} = 90\%$$

> ⚠️ Accuracy is misleading for imbalanced datasets. Use F1 / PR-AUC for fraud.

---

### Q5 · Scaling for KNN

> ✅ **StandardScaler** — KNN is distance-based; income ($10k–$200k) dominates without scaling

| Scaler | Use when |
|---|---|
| StandardScaler | General purpose, no extreme outliers |
| MinMaxScaler | Bounded output needed |
| RobustScaler | Significant outliers present |

---

### Q6 · Random Forest convergence

> ✅ **Model has reached convergence — additional trees give diminishing returns**

- RF **cannot overfit** by adding more trees (unlike GBM)
- More trees → lower variance, but improvement flattens after ~200-300
- Feature importances become **more stable** with more trees (not less)

---

### Q7 · SVM underfitting with linear kernel

> ✅ **Change kernel from linear to RBF** — linear kernel cannot fit non-linear boundaries

| Kernel | Decision boundary |
|---|---|
| `linear` | Straight hyperplane |
| `rbf` ✅ | Non-linear curves — **general purpose default** |
| `poly` | Polynomial curves |

---

### Q8 · Hierarchical clustering — chaining effect

> ✅ **Switch from single linkage to complete linkage**

| Linkage | Distance measure | Cluster shape |
|---|---|---|
| **Single** | Min distance (any 2 points) | Elongated chains ⛓️ ← **problem** |
| **Complete** ✅ | Max distance (farthest points) | Compact, round |
| **Ward** | Minimizes within-cluster variance | Best general choice |

---

### Q9 · Vanishing gradients in RNN

> ✅ **Replace vanilla RNN with LSTM — memory cells + gating mechanisms**

LSTM uses **additive** cell state updates (not multiplicative) → gradients flow unchanged over long sequences.

| Gate | Purpose |
|---|---|
| Forget $f_t$ | What to discard |
| Input $i_t$ | What new info to store |
| Output $o_t$ | What to output |

GRU is a lighter alternative with fewer parameters.

---

### Q10 · Cross-validation for time series

> ✅ **Forward chaining (TimeSeriesSplit)** — always train on past, validate on future

```
Fold 1: Train [t1..t4] → Val [t5]
Fold 2: Train [t1..t5] → Val [t6]
```

K-fold with random shuffling = **data leakage** (future leaks into training).

---

### Q11 · ML REST API — production requirements

> ✅ **FastAPI + Pydantic** — input validation + exception handling + async scalability

| Framework | Missing |
|---|---|
| Basic Flask | No validation, no async |
| Flask + manual validation | Verbose, error-prone |
| **FastAPI + Pydantic** ✅ | Auto validation, async, auto-docs |
| Django REST | Overkill for ML endpoints, no async |

---

### Q12 · 0.1% fraud imbalance — asymmetric cost

> ✅ **SMOTE + cost-sensitive learning (higher penalty for false negatives)**

- **SMOTE**: synthetic minority samples → richer signal than duplication
- **Cost-sensitive**: penalises missed fraud more → model prioritises recall
- **Metric**: PR-AUC, Recall — NOT accuracy

---

### Q13 · XGBoost vs LightGBM architecture

> ✅ **LightGBM: leaf-wise (best-first) growth; XGBoost: level-wise**

| | XGBoost | LightGBM |
|---|---|---|
| Tree growth | Level-wise | Leaf-wise ⚡ |
| Speed on large data | Slower | Faster |
| Memory | Higher | Lower (histogram binning) |
| Overfitting risk | Lower | Higher (use `min_data_in_leaf`) |

---

### Q14 · Transformer: $O(n^2)$ attention for long sequences

> ✅ **Sparse attention — sliding window + global tokens (Longformer)**

Standard: $O(n^2)$ → 50,000² = 2.5 billion weights  
Longformer: $O(n \cdot w)$ → local window + global tokens

- Gradient checkpointing helps memory but doesn't fix $O(n^2)$ complexity
- Chunking breaks long-range dependencies

---

### Q15 · 100k req/s at 50ms latency — ML serving

> ✅ **Distributed system: Ray Serve / Triton + multiple GPU clusters + caching + failover**

$$100{,}000 \text{ req/s} \times 50\text{ms} = 5{,}000 \text{ GPU-equivalents needed}$$

| Feature | Solves |
|---|---|
| Multiple GPU clusters | Throughput |
| Load balancing | Even distribution |
| Caching (Redis) | Cuts GPU work 80%+ for popular items |
| Failover | Reliability SLA |

Serverless → cold start (100ms–1s) violates 100ms SLA.